# 2차시 ― 머신러닝·딥러닝 모델, 직접 써 보기

> 1차시에서 Google Colab에 접속해 코드 셀을 실행해 보았습니다. 이번 노트북은 **파이썬 문법을 몰라도** 괜찮습니다. 문법은 3차시에 배웁니다. 지금은 **잘 만들어진 AI 모델을 한두 줄로 불러와 결과를 직접 확인**하는 것이 목표입니다.
>
> 코드를 한 줄씩 이해하지 못해도 됩니다. **셀을 실행하고(Shift + Enter), 나온 결과를 감상**하세요. 우리가 방금 부른 것이 바로 머신러닝·딥러닝 모델입니다.
>
> **실행 방법**: 코드 셀을 클릭하고 `Shift + Enter`. 위에서 아래로 순서대로 실행하세요.

**오늘 써 볼 모델들**
1. 감성 분석 — 문장이 긍정인지 부정인지
2. 텍스트 요약 — 긴 글을 짧게
3. 번역 — 한국어를 영어로
4. 이미지 분류 — 사진 속 사물이 무엇인지
5. 빈칸 채우기 — 문장의 `[MASK]` 자리에 들어갈 말
6. 제로샷 분류 — 내가 정한 분류 항목으로 글을 나누기


> 📌 **✏️ 콜아웃으로 표시된 "연습문제"는 직접 입력하는 칸입니다.** 강사와 함께 한 줄씩 따라 입력해 봅니다.

## 0. 준비 — 라이브러리 설치

Hugging Face의 `transformers`는 전 세계 연구자들이 학습해 공개한 AI 모델을 손쉽게 불러오는 도구이고, `torch`는 그 모델이 계산을 수행하는 엔진입니다. 아래 셀을 한 번만 실행해 두 도구를 설치합니다. (한두 분 걸릴 수 있습니다.)

> Colab에서 `!` 로 시작하는 줄은 파이썬이 아니라 설치 명령입니다. 지금은 의미를 몰라도 됩니다.


In [ ]:
!pip install -q transformers torch


모델을 불러오는 도구를 가져옵니다. `pipeline` 하나면 "어떤 일(task)을 할지"만 알려 주고 모델을 바로 쓸 수 있습니다.

> 처음 모델을 부를 때는 인터넷에서 모델 파일을 내려받느라 시간이 걸립니다. 두 번째부터는 빨라집니다. 빨간 글씨의 경고(warning)가 보여도 결과가 나오면 정상입니다.


In [ ]:
from transformers import pipeline


## 1. 감성 분석 — 이 문장은 긍정일까, 부정일까

가장 먼저 **문장의 감정**을 읽는 모델입니다. 영화 리뷰처럼 긍정/부정이 담긴 문장을 넣으면, 모델이 `POSITIVE`(긍정) 또는 `NEGATIVE`(부정)와 **확신 정도(score, 0~1)** 를 함께 돌려줍니다.

`task="sentiment-analysis"` 한 줄로 모델이 준비됩니다.


In [ ]:
classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

result = classifier("I really love this movie. It was fantastic!")
print(result)


결과는 `[{'label': 'POSITIVE', 'score': 0.99...}]` 처럼 나옵니다. `label`이 모델의 판단, `score`가 그 판단에 대한 확신입니다. 한 번에 여러 문장을 넣어 비교할 수도 있습니다.


In [ ]:
texts = [
    "The food was delicious and the staff were very kind.",
    "This is the worst experience I have ever had.",
]
for text in texts:
    print(text, "->", classifier(text))


> 이 모델은 영어로 학습되었습니다. 한국어 문장은 뒤의 **3. 번역** 에서 영어로 바꿔 넣어 볼 수 있습니다.


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 1 내 문장으로 감성 분석</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>my_text</code> 의 따옴표 안에 <b>영어 문장</b>을 직접 써서 긍정/부정을 확인하세요.<br>
<b>힌트</b> &nbsp; 칭찬·불평을 섞으면 모델이 어떻게 판단하는지 비교돼요.<br>
<b>예상</b> &nbsp; <code>[{'label': 'POSITIVE'/'NEGATIVE', 'score': ...}]</code>
</div>
</div>

In [ ]:
# 연습문제 1: 아래 따옴표 안 영어 문장을 자유롭게 바꿔 결과를 확인하세요.
my_text = "____여기에 영어 문장을 적으세요____"
print(classifier(my_text))


## 2. 텍스트 요약 — 긴 글을 짧게

이번에는 **긴 글의 핵심만 추려 주는** 모델입니다. `task="summarization"` 으로 부르면, 문단을 넣었을 때 짧은 요약문을 만들어 줍니다. (영어 글을 요약합니다.)


In [ ]:
summarizer = pipeline(
    task="summarization",
    model="sshleifer/distilbart-cnn-12-6"
)


요약할 긴 글을 하나 준비합니다. `article` 안의 영어 문단을 모델에 넣습니다. `max_length`·`min_length` 는 요약문의 대략적인 길이입니다.


In [ ]:
article = (
    "Artificial intelligence is transforming many industries around the world. "
    "In healthcare, AI helps doctors detect diseases earlier by analyzing medical images. "
    "In transportation, self-driving systems use AI to recognize roads and other cars. "
    "Many companies now use AI chatbots to answer customer questions quickly. "
    "Experts say that learning the basics of AI will be important for future jobs."
)

summary = summarizer(article, max_length=45, min_length=15, do_sample=False)
print(summary[0]["summary_text"])


원래 다섯 문장이던 글이 한두 문장으로 줄어듭니다. 모델이 어떤 정보를 남기고 어떤 정보를 버렸는지 비교해 보세요.


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 2 내 글 요약하기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 좋아하는 주제의 영어 글 두세 문장을 <code>my_article</code> 에 넣어 요약 결과를 보세요.<br>
<b>힌트</b> &nbsp; 문장이 길수록 요약이 잘 됩니다(스포츠·영화·취미 등).<br>
<b>예상</b> &nbsp; 여러 문장이 한두 문장으로 줄어듭니다.
</div>
</div>

In [ ]:
# 연습문제 2: 아래 my_article 안 영어 문단을 바꾼 뒤 요약 결과를 확인하세요.
my_article = (
    "____여기에 두세 문장 이상의 영어 글을 적으세요____"
)
print(summarizer(my_article, max_length=45, min_length=15, do_sample=False)[0]["summary_text"])


## 3. 번역 — 한국어를 영어로

`task="translation"` 모델은 한 언어를 다른 언어로 바꿉니다. 여기서는 **한국어 → 영어** 모델(`Helsinki-NLP/opus-mt-ko-en`)을 씁니다.


In [ ]:
translator = pipeline(
    task="translation",
    model="Helsinki-NLP/opus-mt-ko-en"
)

result = translator("오늘 날씨가 정말 좋아서 기분이 좋습니다.")
print(result[0]["translation_text"])


이렇게 번역한 한국어 문장을 **1번 감성 분석 모델에 그대로 이어서** 넣어 볼 수도 있습니다. 한국어로 시작해도 영어 모델로 감정을 읽을 수 있게 되는 셈입니다.


In [ ]:
korean = "이 영화는 정말 최악이었어요. 시간이 아까웠습니다."
english = translator(korean)[0]["translation_text"]
print("번역:", english)
print("감성:", classifier(english))


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 3 한국어를 영어로 번역</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>my_sentence</code> 에 한국어 문장을 넣어 영어 번역을 확인하세요.<br>
<b>힌트</b> &nbsp; 번역 결과를 다시 <code>classifier</code> 에 넣으면 감정까지 볼 수 있어요.<br>
<b>예상</b> &nbsp; 한국어 → 자연스러운 영어 문장
</div>
</div>

In [ ]:
# 연습문제 3: 아래 my_sentence 의 한국어 문장을 바꿔 영어 번역을 확인하세요.
my_sentence = "____여기에 한국어 문장을 적으세요____"
print(translator(my_sentence)[0]["translation_text"])


## 4. 이미지 분류 — 사진 속 사물 알아맞히기

지금까지는 글이었지만, AI는 **그림**도 봅니다. `task="image-classification"` 모델은 사진을 넣으면 "무엇이 찍혔는지"를 후보와 확률로 알려 줍니다. 여기서는 ViT라는 이미지 모델(`google/vit-base-patch16-224`)을 씁니다.

먼저 인터넷에 있는 예시 사진을 불러옵니다. (고양이 두 마리가 있는, Hugging Face가 예제로 제공하는 사진입니다.)


In [ ]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image   # 셀 마지막 줄에 변수 이름만 두면 Colab이 그림을 보여 줍니다


이제 이 사진을 분류 모델에 넣습니다. `top_k=3` 은 가장 가능성 높은 후보 3개를 보겠다는 뜻입니다.


In [ ]:
image_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224"
)

predictions = image_classifier(image, top_k=3)
for p in predictions:
    print(p["label"], "->", round(p["score"], 3))


`label` 은 모델이 추측한 사물 이름, 옆 숫자는 확신 정도입니다. 모델이 사진을 보고 1순위로 무엇을 골랐는지 확인해 보세요.


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 4 다른 사진 분류하기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>my_url</code> 에 다른 사진 주소를 넣어 무엇이 찍혔는지 맞혀 보세요.<br>
<b>힌트</b> &nbsp; 사진 <b>마우스 오른쪽 클릭 → 이미지 주소 복사</b> · 주소가 <code>.jpg</code>/<code>.png</code> 로 끝나야 해요.<br>
<b>예상</b> &nbsp; 후보 3개와 확신 점수(0~1)
</div>
</div>

In [ ]:
# 연습문제 4: my_url 에 다른 사진 주소(.jpg/.png 로 끝나는 URL)를 넣어 분류해 보세요.
my_url = "____여기에 이미지 주소를 붙여넣으세요____"
my_image = Image.open(requests.get(my_url, stream=True).raw)
for p in image_classifier(my_image, top_k=3):
    print(p["label"], "->", round(p["score"], 3))


## 5. 빈칸 채우기 — [MASK] 자리에 들어갈 말

`task="fill-mask"` 모델은 문장에서 가려진 한 칸(`[MASK]`)에 **어떤 단어가 들어갈지** 확률 순으로 제안합니다. 여기서는 한국어 모델(`klue/bert-base`)을 씁니다. 빈칸 자리에 반드시 `[MASK]` 라고 적어야 합니다.


In [ ]:
unmasker = pipeline(
    task="fill-mask",
    model="klue/bert-base"
)

results = unmasker("저는 오늘 아침에 [MASK]을 마셨습니다.")
for r in results[:3]:
    print(r["token_str"], "->", round(r["score"], 3))


모델이 `[MASK]` 자리에 어울리는 단어를 확률과 함께 제안합니다. 빈칸 앞뒤 문맥을 바꾸면 제안되는 단어도 달라집니다.


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 5 빈칸 채우기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>my_sentence</code> 의 한국어 문장을 바꿔 <code>[MASK]</code> 자리에 들어갈 단어를 확인하세요.<br>
<b>힌트</b> &nbsp; 빈칸 자리에는 반드시 <code>[MASK]</code> 를 그대로 두어야 합니다.<br>
<b>예상</b> &nbsp; 어울리는 단어 3개와 점수
</div>
</div>

In [ ]:
# 연습문제 5: 문장을 바꿔 보세요. 단, 빈칸 자리에는 꼭 [MASK] 를 남겨 두세요.
my_sentence = "한경국립대학교는 [MASK]에 있습니다."
for r in unmasker(my_sentence)[:3]:
    print(r["token_str"], "->", round(r["score"], 3))


## 6. 제로샷 분류 — 분류 항목을 내가 정한다

마지막은 **내가 직접 분류 항목(라벨)을 정해** 주면, 따로 학습하지 않은 항목인데도 문장을 그 항목으로 나눠 주는 모델입니다. 이것을 제로샷(zero-shot, 사전 예시 없이) 분류라고 합니다. 한국어도 되는 모델(`joeddav/xlm-roberta-large-xnli`)을 씁니다.


In [ ]:
zero_shot = pipeline(
    task="zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

sentence = "이번 주말에 가족과 함께 제주도로 여행을 떠납니다."
labels = ["여행", "음식", "스포츠", "정치"]

result = zero_shot(sentence, candidate_labels=labels)
for label, score in zip(result["labels"], result["scores"]):
    print(label, "->", round(score, 3))


모델이 내가 준 항목들 중에서 문장에 가장 잘 맞는 것을 1순위로 올려 줍니다(`labels` 는 점수가 높은 순으로 정렬됩니다). 같은 문장이라도 **항목 후보를 바꾸면** 결과가 달라집니다.


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 6 내가 정한 항목으로 분류</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>my_sentence</code> 와 <b>분류 항목 3개</b>(<code>my_labels</code>)를 정해 넣어 보세요.<br>
<b>힌트</b> &nbsp; 후기 문장에 <code>["칭찬","불만","질문"]</code> 처럼 항목을 정하면 사전 예시 없이 골라 줍니다.<br>
<b>예상</b> &nbsp; 항목별 점수(높은 순 정렬)
</div>
</div>

In [ ]:
# 연습문제 6: 문장과 분류 항목(라벨)을 모두 자유롭게 바꿔 보세요.
my_sentence = "____여기에 한국어 문장을 적으세요____"
my_labels = ["____항목1____", "____항목2____", "____항목3____"]

result = zero_shot(my_sentence, candidate_labels=my_labels)
for label, score in zip(result["labels"], result["scores"]):
    print(label, "->", round(score, 3))


## 7. 질문 답하기 (Question Answering)
**지문(context)** 을 주고 **질문(question)** 을 하면, 지문 안에서 답을 찾아 줍니다. (영어 모델)

In [ ]:
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

In [ ]:
context = "Hugging Face is a company based in New York City. It is best known for the Transformers library, which makes AI models easy to use."
qa(question="Where is Hugging Face based?", context=context)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 7 질문 바꿔 답 찾기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>question</code> 을 다른 질문으로 바꿔 지문에서 답을 찾게 하세요.<br>
<b>힌트</b> &nbsp; 답이 <b>지문(context) 안에</b> 있는 질문으로 만드세요.<br>
<b>예상</b> &nbsp; 지문에서 뽑아낸 짧은 답
</div>
</div>

In [ ]:
# 연습문제 7: 질문을 바꿔 실행해 보세요.
qa(question="What is Hugging Face known for?", context=context)

## 8. 글 이어쓰기 (Text Generation)
앞부분을 주면 모델이 **뒤를 이어서** 써 줍니다. 작은 모델이라 가볍습니다. (영어 모델)

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

In [ ]:
generator("In the future, artificial intelligence will",
          max_new_tokens=30, num_return_sequences=1)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 8 글 이어쓰기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 따옴표 안 영어 첫 문장을 바꿔, 모델이 뒤를 어떻게 잇는지 보세요.<br>
<b>힌트</b> &nbsp; 짧은 시작 문장이 무난합니다(예: <code>"My favorite food is"</code>).<br>
<b>예상</b> &nbsp; 시작 문장에 이어지는 영어 문장
</div>
</div>

In [ ]:
# 연습문제 8: 시작 문장을 바꿔 보세요.
generator("My favorite food is", max_new_tokens=30, num_return_sequences=1)

## 9. 개체명 인식 (NER)
문장에서 **사람·장소·기관** 같은 이름을 찾아 표시합니다. (영어 모델)

In [ ]:
ner = pipeline("token-classification", model="dslim/bert-base-NER",
               aggregation_strategy="simple")

In [ ]:
ner("My name is Clara and I work at Google in California.")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 9 이름 찾아내기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 사람·회사·지역 이름을 바꾼 영어 문장으로 실행해 무엇을 찾아내는지 보세요.<br>
<b>힌트</b> &nbsp; <code>PER</code>(사람)·<code>ORG</code>(기관)·<code>LOC</code>(장소) 라벨이 붙습니다.<br>
<b>예상</b> &nbsp; 찾은 이름과 종류 목록
</div>
</div>

In [ ]:
# 연습문제 9: 문장을 바꿔 보세요.
ner("Elon Musk founded SpaceX and Tesla in the United States.")

## 10. 객체 탐지 (Object Detection)
사진 속 물체를 찾아 **네모(박스)와 이름**으로 알려 줍니다. 시각적이라 재밌어요.

In [ ]:
from PIL import Image
import requests
detector = pipeline("object-detection", model="facebook/detr-resnet-50")

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"   # 고양이 두 마리 사진
image = Image.open(requests.get(url, stream=True).raw)
results = detector(image)
for r in results[:6]:
    print(round(r["score"], 2), r["label"], r["box"])

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 10 다른 사진에서 물체 찾기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>my_url</code> 에 다른 사진 주소를 넣어 물체를 찾아 보세요.<br>
<b>힌트</b> &nbsp; 주소가 <code>.jpg</code>/<code>.png</code> 로 끝나야 합니다.<br>
<b>예상</b> &nbsp; 물체 이름 + 점수 + 박스 좌표
</div>
</div>

In [ ]:
# 연습문제 10: 다른 이미지 주소로 바꿔 보세요.
my_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = Image.open(requests.get(my_url, stream=True).raw)
for r in detector(img)[:6]:
    print(round(r["score"], 2), r["label"], r["box"])

## 11. 자주 만나는 오류와 해결
빨간 오류 메시지가 떠도 당황하지 마세요. HF 모델 실습에서 거의 다 아래에 해당합니다.

| 증상 | 원인 | 해결 |
|---|---|---|
| 첫 실행이 수십 초~몇 분 걸림 | 모델을 **처음 내려받는 중** | 정상입니다. 한 번 받으면 다음부턴 빨라요 |
| `NameError: ... pipeline ...` | 위쪽 `import` 셀을 **건너뜀** | 맨 위(0번)부터 순서대로 실행 |
| `ModuleNotFoundError: transformers` | 0번 **설치 셀 미실행** | `!pip install -q transformers torch` 먼저 |
| 이미지 분류에서 오류 | URL이 사진(`.jpg`/`.png`)이 아니거나 깨짐 | 다른 이미지 주소로 교체 |
| 점점 느려지거나 메모리 부족 | 큰 모델을 여러 개 로드 | 런타임 → 세션 다시 시작 후 필요한 셀만 |

> 모델 다운로드에는 인터넷이 필요합니다(Colab은 온라인 환경). 오류 메시지의 **맨 아랫줄**이 핵심이에요.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 종합 미션 직접 바꿔 보기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">종합</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>미션 1</b> &nbsp; 가장 흥미로운 task 하나를 골라 입력 문장을 <b>내 문장</b>으로 바꿔 3번 이상 실행<br>
<b>미션 2 (모델 잇기)</b> &nbsp; 3번 <b>번역</b>(한→영) 결과를 1번 <b>감성 분석</b>에 넣어 한국어 문장의 긍·부정 판별<br>
<b>미션 3</b> &nbsp; 6번 <b>제로샷 분류</b>의 라벨을 내 주제로 변경(예: 뉴스 = 정치·경제·스포츠·연예)<br>
<b>미션 4</b> &nbsp; 5번 <b>빈칸 채우기</b>에서 <code>[MASK]</code> 위치를 옮겨 제안 단어 변화 관찰
</div>
</div>

### ✅ 셀프 체크리스트
- [ ] 여러 pipeline(이제 10가지!)을 실행해 봤다
- [ ] `task` 이름만 바꾸면 다른 일을 시킬 수 있다는 걸 확인했다
- [ ] 입력을 바꾸면 결과가 바뀌는 걸 봤다
- [ ] 번역 → 감성처럼 모델을 **이어 붙여** 봤다
- [ ] 파이썬을 몰라도 '복붙'으로 AI 모델을 써봤다 (문법은 3차시에)

## 13. (보너스) Hugging Face 둘러보기
오늘 쓴 모델들은 모두 **Hugging Face**라는 곳에 공개된 것들입니다. 설치도 코드도 없이 브라우저로 둘러보세요.

- 🤗 **모델 허브** — https://huggingface.co/models · 수십만 개 공개 모델을 task별로 검색. 오늘 쓴 pipeline의 기본 모델도 여기 있어요.
- 🚀 **Spaces** — https://huggingface.co/spaces · 코드 없이 브라우저에서 바로 돌려보는 데모 모음. 마음에 드는 걸 하나 열어 체험해 보세요.

> 오늘 우리가 한 일은 "남이 올려둔 모델을 **가져다 쓰기**"입니다. **3차시**에 파이썬을 배우면, 이 모델들을 내 데이터로 직접 다루게 됩니다.

## 정리 — 우리가 방금 한 일

오늘은 코드 **한두 줄**로 여섯 가지 AI 모델을 불러 직접 결과를 확인했습니다.

| 한 일 | task 이름 | 사용한 모델 |
|---|---|---|
| 감성 분석 | `sentiment-analysis` | distilbert-base-uncased-finetuned-sst-2-english |
| 텍스트 요약 | `summarization` | sshleifer/distilbart-cnn-12-6 |
| 번역(한→영) | `translation` | Helsinki-NLP/opus-mt-ko-en |
| 이미지 분류 | `image-classification` | google/vit-base-patch16-224 |
| 빈칸 채우기 | `fill-mask` | klue/bert-base |
| 제로샷 분류 | `zero-shot-classification` | joeddav/xlm-roberta-large-xnli |

방금 `pipeline` 한 줄로 불러온 것은 **수억 개의 숫자(파라미터)로 학습된 신경망**입니다. 우리가 한 일은 그 거대한 모델에게 문장이나 사진을 건네고 답을 받은 것뿐입니다.

- **4차시**에는 이런 모델을 (작게나마) **직접 만들어** 봅니다.
- **5차시**에는 모델을 우리 용도에 맞게 **직접 미세조정(fine-tuning)** 하고, **챗봇**까지 만들어 봅니다.

오늘 "결과가 신기하다"고 느꼈다면, 그 다음 질문은 "이게 어떻게 가능할까?"입니다. 그 답을 다음 시간들에서 하나씩 풀어 갑니다.
